# FINOVA AI | Financial Well-Being Score
### One-click training notebook • CFPB National Financial Well-Being Survey 2016

**Workflow:** dataset checks → descriptive analysis → split → leakage-safe preprocessing → 5 model comparisons → final held-out testing → charts and insights → saved model.

✅ **To run:** open this notebook in VS Code/Jupyter and select **Run All**. It installs missing Python packages when necessary. The cleaned CSV is already included in `data/`.

> **Important:** This predicts the CFPB **survey-based financial well-being score** (`FWBscore`), not a validated transaction-based global financial health score, credit score, or financial recommendation engine. Survey responses contain categorical response codes, not bank balances or live transactions. The demo is **research-only** and the dataset is from the **United States, 2016**.

**Dataset attribution:** U.S. Consumer Financial Protection Bureau (CFPB), *National Financial Well-Being Survey*, public use file.  
Source: https://www.consumerfinance.gov/data-research/financial-well-being-survey-data/  
Official CSV: https://files.consumerfinance.gov/f/documents/NFWBS_PUF_2016_data.csv

## 0. Package check and imports

The first cell checks the environment and installs missing dependencies using the currently selected Python interpreter.

In [ ]:
import sys, subprocess, importlib.util

DEPENDENCIES = {
    "numpy": "numpy>=1.24",
    "pandas": "pandas>=2.0",
    "sklearn": "scikit-learn>=1.3",
    "matplotlib": "matplotlib>=3.7",
    "joblib": "joblib>=1.3",
    "xgboost": "xgboost>=2.0",
    "lightgbm": "lightgbm>=4.0",
    "catboost": "catboost>=1.2",
}
missing = [spec for module, spec in DEPENDENCIES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing dependencies:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All dependencies already available.")

from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
# Helper module ships inside this ZIP folder; support common VS Code working dirs.
for _base in [Path.cwd(), Path.cwd() / "FINOVA_Financial_Health_Score",
              Path.cwd().parent / "FINOVA_Financial_Health_Score"]:
    if (_base / "compat_models.py").is_file():
        sys.path.insert(0, str(_base.resolve()))
        break
from compat_models import CompatibleCatBoostRegressor

pd.set_option("display.max_columns", 40)
RANDOM_STATE = 42
THREADS = 4
print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__, "| NumPy:", np.__version__)

All dependencies already available.


## 1. Read the correct dataset

The **Baseline** CSV contains respondent records. The *Audit* CSV is **not** for ML training. File detection supports opening this notebook either from the ZIP folder or from its parent directory.

In [ ]:
FILENAME = "FINOVA_Financial_Wellbeing_Baseline.csv"
search_dirs = [Path.cwd(), Path.cwd() / "data",
               Path.cwd() / "FINOVA_Financial_Health_Score" / "data",
               Path.cwd().parent / "data", Path.cwd().parent]
candidates = [folder / FILENAME for folder in search_dirs]
DATA_PATH = next((path for path in candidates if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Baseline CSV missing. Extract the entire ZIP and open the notebook "
        "inside FINOVA_Financial_Health_Score. Searched:\n" +
        "\n".join(str(p) for p in candidates)
    )
df = pd.read_csv(DATA_PATH)
assert df.shape == (6389, 33), f"Unexpected shape: {df.shape}"
assert df["PUF_ID"].is_unique, "Respondent identifier must be unique"
assert df["FWBscore"].notna().all(), "Missing target scores"
assert df["FWBscore"].between(0, 100).all(), "Invalid score values"
print("Loaded:", DATA_PATH.resolve())
print("Rows:", len(df), "| Columns:", len(df.columns))
print("Missing cells:", int(df.isna().sum().sum()))
print("Duplicated rows:", int(df.duplicated().sum()))
print("FWBscore mean:", round(df["FWBscore"].mean(), 2))
display(df.head(5))

## 2. Data quality, score distribution, and features

The baseline has missing survey responses represented as NaN; these will be handled **inside** the training pipeline. We do **not** fill missing values before splitting.

In [ ]:
TARGET = "FWBscore"
NON_FEATURES = ["PUF_ID", "FWBscore", "finalwt"]
feature_cols = [name for name in df.columns if name not in NON_FEATURES]
assert len(feature_cols) == 30, f"Expected 30 predictors, found {len(feature_cols)}"
assert not any(c.startswith(("FWB1_", "FWB2_")) for c in feature_cols),     "Direct target-scale questions must not be used as predictors"
print("Input features:", len(feature_cols))
print("Target:", TARGET)
print("Score summary:", df[TARGET].describe().round(2))
missing_table = (df[feature_cols].isna().sum().sort_values(ascending=False)
                 .rename("missing_cells").to_frame())
missing_table["missing_pct"] = (missing_table["missing_cells"] / len(df) * 100).round(2)
display(missing_table.head(15))

OUTPUT_DIR = Path.cwd() / "outputs"
MODEL_DIR = Path.cwd() / "models"
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.hist(df[TARGET], bins=32, color="#6366f1", edgecolor="white", alpha=0.88)
ax.set(title="CFPB Financial Well-Being Score Distribution", xlabel="FWBscore", ylabel="Respondents")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "score_distribution.png", dpi=150)
plt.show()

## 3. Lock a 70/15/15 split

- **Training (70%)**: learns patterns.
- **Validation (15%)**: compares model families and selects a winner.
- **Test (15%)**: held out until the winner is selected, then evaluated only once.

Approximate score-distribution stratification is used for comparable splits. `PUF_ID` and survey weight `finalwt` are **never** predictive inputs.

In [ ]:
X = df[feature_cols].copy()
y = df[TARGET].astype(float)
weights = df["finalwt"].astype(float)
score_bins = pd.qcut(y, q=10, duplicates="drop")
X_train, X_rest, y_train, y_rest = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=score_bins
)
rest_bins = pd.qcut(y_rest, q=10, duplicates="drop")
X_val, X_test, y_val, y_test = train_test_split(
    X_rest, y_rest, test_size=0.5,
    random_state=RANDOM_STATE, stratify=rest_bins
)
w_train = weights.loc[X_train.index]
w_val = weights.loc[X_val.index]
w_test = weights.loc[X_test.index]
assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)
print(f"Train: {len(X_train):,} | Validation: {len(X_val):,} | Test: {len(X_test):,}")
print("Mean scores:", {"train":round(y_train.mean(),2),
                       "val":round(y_val.mean(),2), "test":round(y_test.mean(),2)})

## 4. Categorical preprocessing (leakage-safe)

Most survey fields represent **coded categories** rather than money. We treat all 30 as categorical for a transparent baseline. The imputer and one-hot encoder are fitted **only on training rows**, wrapped in the same pipeline as each model. Unknown categories at inference time are safely ignored.

In [ ]:
# Numeric codes are treated as categories, not as continuous money amounts.
def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("cat", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="constant", fill_value=-999.0)),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]), feature_cols)
        ],
        remainder="drop"
    )

def make_pipeline(regressor):
    return Pipeline(steps=[
        ("preprocess", make_preprocessor()),
        ("model", clone(regressor)),
    ])

candidate_models = {
    "Mean Baseline": DummyRegressor(strategy="mean"),
    "Ridge Regression": Ridge(alpha=10.0),
    "XGBoost": XGBRegressor(
        n_estimators=350, max_depth=4, learning_rate=0.045,
        subsample=0.85, colsample_bytree=0.90,
        objective="reg:squarederror", tree_method="hist",
        n_jobs=THREADS, random_state=RANDOM_STATE,
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=350, num_leaves=20, max_depth=-1,
        learning_rate=0.035, subsample=0.85,
        colsample_bytree=0.90, min_child_samples=35,
        verbosity=-1, n_jobs=THREADS, random_state=RANDOM_STATE,
    ),
    "CatBoost": CompatibleCatBoostRegressor(
        iterations=350, depth=5, learning_rate=0.055,
        thread_count=THREADS, random_seed=RANDOM_STATE,
    ),
}
print("Models:", list(candidate_models))

## 5. Train 5 baseline and advanced models

Selection criterion: **validation MAE** (lower is better). This is an efficient first comparison, **not** exhaustive hyperparameter tuning.

In [ ]:
validation_rows = []
for name, base_model in candidate_models.items():
    pipeline = make_pipeline(base_model)
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_val)
    row = {
        "Model": name,
        "Val_MAE": mean_absolute_error(y_val, pred),
        "Val_RMSE": np.sqrt(mean_squared_error(y_val, pred)),
        "Val_R2": r2_score(y_val, pred),
        "Val_Weighted_MAE": mean_absolute_error(y_val, pred, sample_weight=w_val),
    }
    validation_rows.append(row)
    print(f"{name:18s} | MAE {row['Val_MAE']:.4f} | RMSE {row['Val_RMSE']:.4f} | R² {row['Val_R2']:.4f}")

comparison = pd.DataFrame(validation_rows).sort_values("Val_MAE").reset_index(drop=True)
comparison.to_csv(OUTPUT_DIR / "model_validation_comparison.csv", index=False)
print("\nValidation comparison (lower MAE is better):")
display(comparison.round(4))
BEST_NAME = comparison.iloc[0]["Model"]
print("Selected winner:", BEST_NAME)

## 6. Plot validation comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.4))
ax.barh(comparison["Model"], comparison["Val_MAE"], color="#6366f1")
ax.invert_yaxis()
ax.set(xlabel="Validation MAE — lower is better", title="FINOVA AI: Model Comparison")
ax.grid(axis="x", alpha=0.2)
for i, value in enumerate(comparison["Val_MAE"]):
    ax.text(value + 0.12, i, f"{value:.2f}", va="center", fontsize=9)
ax.set_xlim(0, comparison["Val_MAE"].max() * 1.18)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "validation_model_comparison.png", dpi=160)
plt.show()

## 7. Fit winner on train + validation, then evaluate **once** on test

Now that model selection is complete, the winner is trained from scratch on training+validation rows. The final test data did not inform model choice.

In [ ]:
X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])
best_pipeline = make_pipeline(candidate_models[BEST_NAME])
best_pipeline.fit(X_trainval, y_trainval)
test_pred = np.clip(best_pipeline.predict(X_test), 0, 100)

test_metrics = {
    "selected_model": BEST_NAME,
    "test_mae": float(mean_absolute_error(y_test, test_pred)),
    "test_rmse": float(np.sqrt(mean_squared_error(y_test, test_pred))),
    "test_r2": float(r2_score(y_test, test_pred)),
    "test_weighted_mae": float(mean_absolute_error(y_test, test_pred, sample_weight=w_test)),
    "train_rows": len(X_train),
    "validation_rows": len(X_val),
    "test_rows": len(X_test),
    "feature_count": len(feature_cols),
}
print("FINAL HELD-OUT TEST RESULTS")
print("Model:", BEST_NAME)
for key in ["test_mae", "test_rmse", "test_r2", "test_weighted_mae"]:
    print(f"{key}: {test_metrics[key]:.4f}")
with (OUTPUT_DIR / "test_metrics.json").open("w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2)
prediction_rows = pd.DataFrame({
    "PUF_ID": df.loc[X_test.index, "PUF_ID"].values,
    "actual_fwbscore": y_test.values,
    "predicted_fwbscore": test_pred,
    "absolute_error": np.abs(y_test.values - test_pred),
})
prediction_rows.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)
display(prediction_rows.head(10).round(2))

## 8. Prediction and error charts

In [ ]:
fig, ax = plt.subplots(figsize=(6.3, 6.0))
ax.scatter(y_test.to_numpy(), test_pred, s=16, alpha=0.38, color="#6366f1")
ax.plot([0, 100], [0, 100], linestyle="--", color="#334155", linewidth=1.5)
ax.set(xlim=(0,100), ylim=(0,100),
       xlabel="Actual FWBscore", ylabel="Predicted FWBscore",
       title=f"Held-out Test: {BEST_NAME}")
ax.grid(alpha=0.15)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "actual_vs_predicted.png", dpi=160)
plt.show()

errors = y_test.to_numpy() - test_pred
fig, ax = plt.subplots(figsize=(8.5,4.2))
ax.hist(errors, bins=35, color="#0d9488", edgecolor="white", alpha=0.9)
ax.axvline(0, color="#334155", linestyle="--")
ax.set(title="Test Residual Distribution", xlabel="Actual − Predicted", ylabel="Rows")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "test_residuals.png", dpi=160)
plt.show()

## 9. Which survey features matter most?

For tree models, this uses native feature importance. For Ridge, it uses absolute coefficients. We aggregate the individual one-hot category columns back into their original survey feature. **Feature importance is association, not causality.**

In [ ]:
trained_estimator = best_pipeline.named_steps["model"]
enc = best_pipeline.named_steps["preprocess"].named_transformers_["cat"].named_steps["onehot"]

if hasattr(trained_estimator, "feature_importances_"):
    expanded_importance = np.asarray(trained_estimator.feature_importances_, dtype=float)
elif hasattr(trained_estimator, "coef_"):
    expanded_importance = np.abs(np.asarray(trained_estimator.coef_)).ravel()
else:
    expanded_importance = None

if expanded_importance is not None:
    cursor = 0
    grouped = []
    for original_col, categories in zip(feature_cols, enc.categories_):
        span = len(categories)
        grouped.append({"feature": original_col,
                        "importance": float(expanded_importance[cursor:cursor+span].sum())})
        cursor += span
    assert cursor == len(expanded_importance)
    importance_df = (pd.DataFrame(grouped)
                     .sort_values("importance", ascending=False))
    importance_df.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)
    display(importance_df.head(15).round(3))
    fig, ax = plt.subplots(figsize=(8.5, 6))
    top = importance_df.head(15).iloc[::-1]
    ax.barh(top["feature"], top["importance"], color="#0d9488")
    ax.set(xlabel="Aggregate importance (model-specific)",
           title=f"Top Feature Groups: {BEST_NAME}")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "feature_importance.png", dpi=160)
    plt.show()
else:
    print("Selected baseline model has no meaningful feature-importance values.")

## 10. Inspect survey subgroup errors

For exploratory quality checks, compare errors by the income-band response `PPINCIMP`. Values are **coded survey categories**, not currency values. Small groups are not interpreted as stable estimates; this is not a complete fairness audit.

In [ ]:
income_slice = pd.DataFrame({
    "income_band_code": X_test["PPINCIMP"].fillna(-999).astype(int).astype(str),
    "absolute_error": np.abs(y_test.to_numpy() - test_pred)
})
slice_report = (income_slice.groupby("income_band_code", as_index=False)
                .agg(n=("absolute_error", "size"), mae=("absolute_error", "mean"))
                .sort_values("income_band_code"))
slice_report["mae"] = slice_report["mae"].round(3)
slice_report.to_csv(OUTPUT_DIR / "income_band_error_analysis.csv", index=False)
display(slice_report)

## 11. Save and reload the trained model

A single `.joblib` file contains the **full preprocessing pipeline and final regression model**. Supply future survey records with the **same 30 coded input columns**, not ordinary bank-transaction amounts. Only load model files from sources you trust.

In [ ]:
MODEL_PATH = MODEL_DIR / "finova_fwb_score_pipeline.joblib"
joblib.dump(best_pipeline, MODEL_PATH, compress=3)
metadata = {
    "project": "FINOVA AI - Financial Well-Being Score (research demo)",
    "target": "FWBscore",
    "target_definition": "CFPB self-reported Financial Well-Being Scale score",
    "data_origin": "CFPB National Financial Well-Being Survey public-use data, 2016",
    "model_name": BEST_NAME,
    "input_features": feature_cols,
    "data_constraints": [
        "Features are survey response codes, not account transactions",
        "U.S. survey 2016; cannot claim global financial validity",
        "No direct FWB1/FWB2 score-construction items among predictors",
        "PUF_ID and finalwt are not input predictors",
        "Model not validated for financial advice, lending, or credit decisions",
    ],
    "test_metrics": test_metrics,
}
with (MODEL_DIR / "model_metadata.json").open("w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

loaded_model = joblib.load(MODEL_PATH)
example_row = X_test.iloc[[0]].copy()
example_prediction = np.clip(loaded_model.predict(example_row), 0, 100)[0]
print("Saved model:", MODEL_PATH.resolve())
print("Sample respondent score prediction:", round(float(example_prediction), 2))
print("Actual survey score:", y_test.iloc[0])
print("Model + metadata saved successfully.")

## ✅ Run All complete

**Generated under `outputs/`:** score distribution PNG, validation comparison CSV/PNG, held-out test metrics JSON, test predictions CSV, prediction/residual plots, feature importance CSV/PNG (when supported), and subgroup error CSV.

**Generated under `models/`:** `finova_fwb_score_pipeline.joblib` and `model_metadata.json`.

**Next FINOVA AI engineering step:** build a *separate* explainable financial-health scoring engine from user income, expenses, liquidity, debt obligations, and savings goals; use appropriate inputs and validation before production release. **Do not** treat the survey regressor as a transaction-based financial health score.